# iPVC Treatment Non-response Prediction — Revised Analysis

**Revisions:** Bug fixes for data leakage, proper train/val/ext split, XGBoost added, PVC sensitivity, recalibration.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                             matthews_corrcoef, confusion_matrix, classification_report,
                             balanced_accuracy_score)
from sklearn.calibration import calibration_curve
import scipy.stats as stats
from scipy.stats import loguniform, uniform
from scipy.special import logit as sp_logit, expit as sp_expit
import joblib

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# XGBoost
from xgboost import XGBClassifier

# SHAP
import shap

# Plotting
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Device setup — works on Mac CPU and Colab GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Output directories
import os
BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
OUT_DIR = os.path.join(BASE_DIR, 'revizyon', 'outputs')
SENS_DIR = os.path.join(OUT_DIR, 'sensitivity')
FIG_DIR = os.path.join(BASE_DIR, 'revizyon', 'figures')
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(SENS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

## Data Loading & Train/Val/External Split

In [ ]:
# Load data
df = pd.read_csv('scriptler_data/df.csv', index_col=0)
print(f"Total dataset: {df.shape}")

# Region coding (Şehir=1 → Istanbul, Şehir=2,3,4 → Anatolia)
df['Şehir'] = df['Şehir'].astype(int)
df['region'] = df['Şehir'].apply(lambda x: 1 if x == 1 else 2)

# Feature definitions
numeric_features = ['PVCyüzdesi', 'PVCQRS', 'LVEF', 'Yaş', 'PVCPrematurındex',
    'QRSratio', 'OrtalamaHR', 'SemptomSüresi', 'QTCsinus',
    'PVCCouplingIntervaldispersiyon', 'CIvariability',
    'PVCPeakQRSduration', 'PVCCouplingInterval', 'PVCCompansatuarInterval']
categorical_features = ['MultifokalPVC', 'Non_susteinedVT', 'Cins', 'HT', 'DM', 'Fullcompansasion']
all_features = numeric_features + categorical_features

# Istanbul / Anatolia split
istanbul = df[df['region'] == 1].copy()
anatolia = df[df['region'] == 2].copy()
print(f"Istanbul: {len(istanbul)}, Anatolia: {len(anatolia)}")

# Istanbul 80/20 train-validation split (for DL early stopping + recalibration)
X_ist = istanbul[all_features].copy()
y_ist = istanbul['Group'].copy()

X_tr, X_val, y_tr, y_val = train_test_split(
    X_ist, y_ist, test_size=0.20, stratify=y_ist, random_state=RANDOM_STATE
)

# Anatolia — completely untouched external test
X_ext = anatolia[all_features].copy()
y_ext = anatolia['Group'].copy()

print(f"\nIstanbul Train: {len(X_tr)}, Istanbul Val: {len(X_val)}, Anatolia External: {len(X_ext)}")
print(f"Train class dist: {y_tr.value_counts().to_dict()}")
print(f"Val class dist:   {y_val.value_counts().to_dict()}")
print(f"Ext class dist:   {y_ext.value_counts().to_dict()}")

## Preprocessing (fit on train only — Major 2 fix)

In [ ]:
# Ensure categorical features are string type for LabelEncoder
for col in categorical_features:
    X_tr[col] = X_tr[col].astype(str)
    X_val[col] = X_val[col].astype(str)
    X_ext[col] = X_ext[col].astype(str)

# LabelEncoder — fit ONLY on train (Major 2 fix)
le_dict = {}
for col in categorical_features:
    le = LabelEncoder()
    X_tr[col] = le.fit_transform(X_tr[col])
    X_val[col] = le.transform(X_val[col])
    X_ext[col] = le.transform(X_ext[col])
    le_dict[col] = le

# StandardScaler — fit ONLY on train
scaler = StandardScaler()
X_tr[numeric_features] = scaler.fit_transform(X_tr[numeric_features])
X_val[numeric_features] = scaler.transform(X_val[numeric_features])
X_ext[numeric_features] = scaler.transform(X_ext[numeric_features])

# Verify: val/ext means should NOT be 0 (since scaler was fit on train)
print("Validation set numeric means (should ≠ 0):")
print(X_val[numeric_features].mean().round(3).to_dict())

# Save scaler + encoders for Gradio
joblib.dump(scaler, os.path.join(OUT_DIR, 'scaler.pkl'))
joblib.dump(le_dict, os.path.join(OUT_DIR, 'label_encoders.pkl'))
print("\nScaler and encoders saved.")

# Store val predictions for recalibration later
val_predictions = {}
ext_predictions = {}

## Logistic Regression

In [ ]:
# Hyperparameter search space
lr_params = [
    {'penalty': ['l1'], 'C': stats.loguniform(1e-5, 100), 'solver': ['saga'],
     'class_weight': ['balanced', None], 'max_iter': [1000], 'random_state': [42]},
    {'penalty': ['elasticnet'], 'C': stats.loguniform(1e-5, 100), 'solver': ['saga'],
     'l1_ratio': stats.uniform(0, 1), 'class_weight': ['balanced', None],
     'max_iter': [1000], 'random_state': [42]},
    {'penalty': ['l2'], 'C': stats.loguniform(1e-5, 100), 'solver': ['saga'],
     'class_weight': ['balanced', None], 'max_iter': [1000], 'random_state': [42]},
]

lr = LogisticRegression()
lr_search = RandomizedSearchCV(lr, lr_params, n_iter=100, cv=5,
                                scoring='roc_auc', random_state=42, n_jobs=-1, verbose=1)
lr_search.fit(X_tr, y_tr)

best_lr = lr_search.best_estimator_
print(f"\nBest LR params: {lr_search.best_params_}")
print(f"Best CV AUC: {lr_search.best_score_:.4f}")

# Validation predictions (for recalibration)
lr_val_proba = best_lr.predict_proba(X_val)[:, 1]
val_predictions['lr'] = lr_val_proba

# External test
lr_ext_proba = best_lr.predict_proba(X_ext)[:, 1]
ext_predictions['lr'] = lr_ext_proba
lr_ext_pred = (lr_ext_proba > 0.5).astype(int)

print(f"\n=== LR External Test ===")
print(f"AUC: {roc_auc_score(y_ext, lr_ext_proba):.4f}")
print(f"Accuracy: {accuracy_score(y_ext, lr_ext_pred):.4f}")
print(f"F1: {f1_score(y_ext, lr_ext_pred):.4f}")
print(f"MCC: {matthews_corrcoef(y_ext, lr_ext_pred):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_ext, lr_ext_pred):.4f}")

# Save results
pd.DataFrame({'True_Labels': y_ext.values, 'Predicted_Probabilities': lr_ext_proba}
).to_csv(os.path.join(OUT_DIR, 'logistic_regression_results.csv'), index=False)

# Save model
joblib.dump(best_lr, os.path.join(OUT_DIR, 'logistic_regression_model.pkl'))

In [ ]:
# LR SHAP Analysis
print("Computing LR SHAP values...")
lr_explainer = shap.LinearExplainer(best_lr, X_tr)
lr_shap_values = lr_explainer.shap_values(X_ext)

plt.figure(figsize=(12, 8))
shap.summary_plot(lr_shap_values, X_ext, feature_names=all_features, show=False, max_display=20)
plt.title('Logistic Regression — SHAP Summary')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'shap_lr.png'), dpi=300, bbox_inches='tight')
plt.close()
print("LR SHAP saved.")

## MLP (Multi-Layer Perceptron)

In [ ]:
mlp_params = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50), (100, 100),
                           (50, 25), (100, 50, 25), (100, 100, 50)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam'],
    'alpha': loguniform(1e-5, 1),
    'learning_rate': ['constant', 'adaptive'],
    'learning_rate_init': loguniform(1e-4, 1e-2),
    'max_iter': [1000],
    'early_stopping': [True],
    'validation_fraction': [0.1],
    'n_iter_no_change': [10],
    'random_state': [42]
}

print("Training MLP...")
mlp = MLPClassifier()
mlp_search = RandomizedSearchCV(mlp, mlp_params, n_iter=50, cv=5,
                                 scoring='roc_auc', random_state=42, n_jobs=-1, verbose=1)
mlp_search.fit(X_tr, y_tr)

best_mlp = mlp_search.best_estimator_
print(f"\nBest MLP params: {mlp_search.best_params_}")
print(f"Best CV AUC: {mlp_search.best_score_:.4f}")

# Validation predictions
mlp_val_proba = best_mlp.predict_proba(X_val)[:, 1]
val_predictions['mlp'] = mlp_val_proba

# External test
mlp_ext_proba = best_mlp.predict_proba(X_ext)[:, 1]
ext_predictions['mlp'] = mlp_ext_proba
mlp_ext_pred = (mlp_ext_proba > 0.5).astype(int)

print(f"\n=== MLP External Test ===")
print(f"AUC: {roc_auc_score(y_ext, mlp_ext_proba):.4f}")
print(f"F1: {f1_score(y_ext, mlp_ext_pred):.4f}")
print(f"MCC: {matthews_corrcoef(y_ext, mlp_ext_pred):.4f}")

pd.DataFrame({'True_Labels': y_ext.values, 'Predicted_Probabilities': mlp_ext_proba}
).to_csv(os.path.join(OUT_DIR, 'mlp_results.csv'), index=False)
joblib.dump(best_mlp, os.path.join(OUT_DIR, 'mlp_model.pkl'))

In [ ]:
print("Computing MLP SHAP values...")
mlp_bg = shap.sample(X_tr, 100, random_state=42)
mlp_explainer = shap.KernelExplainer(best_mlp.predict_proba, mlp_bg, link="logit")
mlp_shap = mlp_explainer.shap_values(X_ext.sample(n=min(100, len(X_ext)), random_state=42), nsamples=100)

plt.figure(figsize=(12, 8))
sv = mlp_shap[1] if isinstance(mlp_shap, list) else mlp_shap
shap.summary_plot(sv, X_ext.sample(n=min(100, len(X_ext)), random_state=42),
                  feature_names=all_features, show=False, max_display=20)
plt.title('MLP — SHAP Summary')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'shap_mlp.png'), dpi=300, bbox_inches='tight')
plt.close()
print("MLP SHAP saved.")

## XGBoost (New — Major 10)

In [ ]:
xgb_params = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'reg_alpha': [0, 0.01, 0.1, 1],
    'reg_lambda': [1, 3, 5, 10],
    'scale_pos_weight': [1, len(y_tr[y_tr==0])/max(len(y_tr[y_tr==1]), 1)]
}

print("Training XGBoost...")
xgb = XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False)
xgb_search = RandomizedSearchCV(xgb, xgb_params, n_iter=100, cv=5,
                                 scoring='roc_auc', random_state=42, n_jobs=-1, verbose=1)
xgb_search.fit(X_tr, y_tr)

best_xgb = xgb_search.best_estimator_
print(f"\nBest XGBoost params: {xgb_search.best_params_}")
print(f"Best CV AUC: {xgb_search.best_score_:.4f}")

# Validation predictions
xgb_val_proba = best_xgb.predict_proba(X_val)[:, 1]
val_predictions['xgboost'] = xgb_val_proba

# External test
xgb_ext_proba = best_xgb.predict_proba(X_ext)[:, 1]
ext_predictions['xgboost'] = xgb_ext_proba
xgb_ext_pred = (xgb_ext_proba > 0.5).astype(int)

print(f"\n=== XGBoost External Test ===")
print(f"AUC: {roc_auc_score(y_ext, xgb_ext_proba):.4f}")
print(f"F1: {f1_score(y_ext, xgb_ext_pred):.4f}")
print(f"MCC: {matthews_corrcoef(y_ext, xgb_ext_pred):.4f}")

pd.DataFrame({'True_Labels': y_ext.values, 'Predicted_Probabilities': xgb_ext_proba}
).to_csv(os.path.join(OUT_DIR, 'xgboost_results.csv'), index=False)
joblib.dump(best_xgb, os.path.join(OUT_DIR, 'xgboost_model.pkl'))

In [ ]:
print("Computing XGBoost SHAP values...")
xgb_explainer = shap.TreeExplainer(best_xgb)
xgb_shap = xgb_explainer.shap_values(X_ext)

plt.figure(figsize=(12, 8))
shap.summary_plot(xgb_shap, X_ext, feature_names=all_features, show=False, max_display=20)
plt.title('XGBoost — SHAP Summary')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'shap_xgboost.png'), dpi=300, bbox_inches='tight')
plt.close()
print("XGBoost SHAP saved.")

## TabTransformer (Bug 2 fix — val_loader instead of test_loader)

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping

# Dataset class
class TabularDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.FloatTensor(X.values if hasattr(X, 'values') else X)
        self.y = torch.LongTensor(y.values if hasattr(y, 'values') else y) if y is not None else None
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

# TabTransformer model
class TabTransformer(pl.LightningModule):
    def __init__(self, input_dim, num_classes=2, d_model=64, nhead=4, num_layers=3, dropout=0.1):
        super().__init__()
        self.save_hyperparameters()
        self.embedding = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=dropout, activation='gelu', batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_classes)
        )
        self.criterion = nn.CrossEntropyLoss()
        self.validation_outputs = []

    def forward(self, x):
        x = self.embedding(x)
        x = x.unsqueeze(1)
        x = self.transformer_encoder(x)
        x = x.squeeze(1)
        return self.fc(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.criterion(self(x), y)
        self.log('train_loss', loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log('val_loss', loss)
        self.validation_outputs.append({
            'val_loss': loss, 'y_true': y,
            'y_pred': torch.softmax(y_hat, dim=1)[:, 1]
        })

    def on_validation_epoch_end(self):
        y_true = torch.cat([x['y_true'] for x in self.validation_outputs]).cpu()
        y_pred = torch.cat([x['y_pred'] for x in self.validation_outputs]).cpu()
        if len(torch.unique(y_true)) > 1:
            auc = roc_auc_score(y_true.numpy(), y_pred.numpy())
            self.log('val_auc', auc)
        self.validation_outputs.clear()

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=1e-4)
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=5)
        return {'optimizer': opt, 'lr_scheduler': sch, 'monitor': 'val_loss'}

# DataLoaders — FIXED: val_loader = Istanbul 20%, NOT Anatolia
train_dataset = TabularDataset(X_tr, y_tr)
val_dataset = TabularDataset(X_val, y_val)      # Istanbul 20% for early stopping
ext_dataset = TabularDataset(X_ext, y_ext)       # Anatolia for final evaluation only

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128)     # ← FIX: was test_loader (Anatolia)
ext_loader = DataLoader(ext_dataset, batch_size=128)

# Train
tt_model = TabTransformer(input_dim=X_tr.shape[1], d_model=64, nhead=4, num_layers=3, dropout=0.1)
early_stop = EarlyStopping(monitor='val_loss', patience=10, mode='min')
trainer = pl.Trainer(
    max_epochs=100,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    callbacks=[early_stop],
    enable_progress_bar=True
)

# FIXED: trainer.fit uses val_loader (Istanbul 20%), NOT ext_loader (Anatolia)
trainer.fit(tt_model, train_loader, val_loader)

# Validation predictions (for recalibration)
tt_model.eval()
tt_val_proba = []
with torch.no_grad():
    for batch in val_loader:
        x, _ = batch
        proba = torch.softmax(tt_model(x), dim=1)[:, 1]
        tt_val_proba.extend(proba.cpu().numpy())
val_predictions['tabtransformer'] = np.array(tt_val_proba)

# External test — single evaluation after training complete
tt_ext_proba = []
with torch.no_grad():
    for batch in ext_loader:
        x, _ = batch
        proba = torch.softmax(tt_model(x), dim=1)[:, 1]
        tt_ext_proba.extend(proba.cpu().numpy())
tt_ext_proba = np.array(tt_ext_proba)
ext_predictions['tabtransformer'] = tt_ext_proba
tt_ext_pred = (tt_ext_proba > 0.5).astype(int)

print(f"\n=== TabTransformer External Test ===")
print(f"AUC: {roc_auc_score(y_ext, tt_ext_proba):.4f}")
print(f"F1: {f1_score(y_ext, tt_ext_pred):.4f}")
print(f"MCC: {matthews_corrcoef(y_ext, tt_ext_pred):.4f}")

pd.DataFrame({'True_Labels': y_ext.values, 'Predicted_Probabilities': tt_ext_proba}
).to_csv(os.path.join(OUT_DIR, 'tabtransformer_results.csv'), index=False)
torch.save(tt_model.state_dict(), os.path.join(OUT_DIR, 'tabtransformer_model.pth'))

In [ ]:
print("Computing TabTransformer SHAP values...")
def tt_predict_proba(X):
    tt_model.eval()
    with torch.no_grad():
        t = torch.FloatTensor(X if isinstance(X, np.ndarray) else X.values)
        return torch.softmax(tt_model(t), dim=1)[:, 1].cpu().numpy()

tt_bg = shap.sample(X_tr, 50, random_state=42)
tt_explainer = shap.KernelExplainer(tt_predict_proba, tt_bg, link="logit")
tt_sample = X_ext.sample(n=min(50, len(X_ext)), random_state=42)
tt_shap = tt_explainer.shap_values(tt_sample, nsamples=100)

plt.figure(figsize=(12, 8))
shap.summary_plot(tt_shap, tt_sample, feature_names=all_features, show=False, max_display=20)
plt.title('TabTransformer — SHAP Summary')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'shap_tabtransformer.png'), dpi=300, bbox_inches='tight')
plt.close()
print("TabTransformer SHAP saved.")

## TabNet (Bug 3 fix — eval_set = Istanbul 20%)

In [ ]:
from pytorch_tabnet.tab_model import TabNetClassifier

tabnet_params = {
    "n_d": 40, "n_a": 40, "n_steps": 3,        # Supplement values (not code defaults 8/8)
    "gamma": 1.3, "n_independent": 2, "n_shared": 2,
    "cat_idxs": [], "cat_dims": [], "cat_emb_dim": [],
    "lambda_sparse": 1e-3,
    "optimizer_fn": torch.optim.Adam,
    "optimizer_params": dict(lr=2e-2),
    "mask_type": "entmax",
    "scheduler_params": dict(mode="min", patience=5, min_lr=1e-5, factor=0.5),
    "scheduler_fn": torch.optim.lr_scheduler.ReduceLROnPlateau,
    "seed": 42, "verbose": 10
}

print("Training TabNet...")
clf_tabnet = TabNetClassifier(**tabnet_params)

# FIXED: eval_set = Istanbul 20% validation, NOT Anatolia external test
clf_tabnet.fit(
    X_train=X_tr.values, y_train=y_tr.values,
    eval_set=[(X_val.values, y_val.values)],  # ← FIX: was X_test (Anatolia)
    eval_name=['val'],
    max_epochs=100,
    patience=25,
    batch_size=128,
    virtual_batch_size=64,
    num_workers=0,
    drop_last=False
)

# Validation predictions
tn_val_proba = clf_tabnet.predict_proba(X_val.values)[:, 1]
val_predictions['tabnet'] = tn_val_proba

# External test
tn_ext_proba = clf_tabnet.predict_proba(X_ext.values)[:, 1]
ext_predictions['tabnet'] = tn_ext_proba
tn_ext_pred = (tn_ext_proba > 0.5).astype(int)

print(f"\n=== TabNet External Test ===")
print(f"AUC: {roc_auc_score(y_ext, tn_ext_proba):.4f}")
print(f"F1: {f1_score(y_ext, tn_ext_pred):.4f}")
print(f"MCC: {matthews_corrcoef(y_ext, tn_ext_pred):.4f}")

pd.DataFrame({'True_Labels': y_ext.values, 'Predicted_Probabilities': tn_ext_proba}
).to_csv(os.path.join(OUT_DIR, 'tabnet_results.csv'), index=False)

# Save TabNet
clf_tabnet.save_model(os.path.join(OUT_DIR, 'tabnet_model'))

In [ ]:
print("Computing TabNet SHAP values...")
tn_bg = shap.sample(X_tr.values, 100, random_state=42)
tn_explainer = shap.KernelExplainer(clf_tabnet.predict_proba, tn_bg)
tn_sample_idx = np.random.RandomState(42).choice(len(X_ext), min(50, len(X_ext)), replace=False)
tn_shap = tn_explainer.shap_values(X_ext.values[tn_sample_idx], nsamples=100)

plt.figure(figsize=(12, 8))
sv = tn_shap[1] if isinstance(tn_shap, list) else tn_shap
shap.summary_plot(sv, X_ext.values[tn_sample_idx],
                  feature_names=all_features, show=False, max_display=20)
plt.title('TabNet — SHAP Summary')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'shap_tabnet.png'), dpi=300, bbox_inches='tight')
plt.close()
print("TabNet SHAP saved.")

## KAN — Kolmogorov-Arnold Network (Bug 4 fix — val-based model selection)

In [ ]:
# KAN Architecture
class KANDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.FloatTensor(X.values if hasattr(X, 'values') else X)
        self.y = torch.LongTensor(y.values if hasattr(y, 'values') else y) if y is not None else None
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return (self.X[idx], self.y[idx]) if self.y is not None else self.X[idx]

class KolmogorovArnoldLayer(nn.Module):
    def __init__(self, input_dim, inner_dim, output_dim):
        super().__init__()
        self.inner_functions = nn.ModuleList([
            nn.Sequential(nn.Linear(1, inner_dim), nn.ReLU(), nn.Linear(inner_dim, 1))
            for _ in range(input_dim)
        ])
        self.outer_function = nn.Sequential(
            nn.Linear(input_dim, inner_dim), nn.ReLU(), nn.Linear(inner_dim, output_dim)
        )
    def forward(self, x):
        inner_outputs = [f(x[:, i:i+1]) for i, f in enumerate(self.inner_functions)]
        return self.outer_function(torch.cat(inner_outputs, dim=1))

class KolmogorovArnoldNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dims=[94, 55], inner_dim=37, dropout=0.467):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hd in hidden_dims:
            layers.append(KolmogorovArnoldLayer(prev_dim, inner_dim, hd))
            prev_dim = hd
        self.kan_layers = nn.ModuleList(layers)
        self.dropout = nn.Dropout(dropout)
        self.output_layer = nn.Linear(hidden_dims[-1], 2)
    def forward(self, x):
        for layer in self.kan_layers:
            x = self.dropout(layer(x))
        return self.output_layer(x)

# DataLoaders
kan_train_ds = KANDataset(X_tr, y_tr)
kan_val_ds = KANDataset(X_val, y_val)
kan_ext_ds = KANDataset(X_ext, y_ext)
kan_train_loader = DataLoader(kan_train_ds, batch_size=32, shuffle=True)
kan_val_loader = DataLoader(kan_val_ds, batch_size=32)
kan_ext_loader = DataLoader(kan_ext_ds, batch_size=32)

# Model setup — using supplement values (94, 55, 37)
kan_model = KolmogorovArnoldNetwork(
    input_dim=X_tr.shape[1], hidden_dims=[94, 55], inner_dim=37, dropout=0.467
).to(device)
kan_criterion = nn.CrossEntropyLoss()
kan_optimizer = optim.Adam(kan_model.parameters(), lr=0.001)
kan_scheduler = optim.lr_scheduler.ReduceLROnPlateau(kan_optimizer, mode='min', factor=0.5, patience=5)

def kan_train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, pred = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (pred == labels).sum().item()
    return total_loss / len(loader), correct / total

def kan_evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_probs, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            probs = torch.softmax(outputs, dim=1)
            _, pred = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
            all_probs.extend(probs[:, 1].cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), correct / total, np.array(all_probs), np.array(all_labels)

# Training loop — FIXED: model selection on VAL set, not test set
print("Training KAN...")
best_val_auc = 0
best_kan_state = None
patience_counter = 0
PATIENCE = 10

for epoch in range(50):
    train_loss, train_acc = kan_train_epoch(kan_model, kan_train_loader, kan_criterion, kan_optimizer)

    # FIXED: evaluate on VALIDATION set (Istanbul 20%), not external test
    val_loss, val_acc, val_probs, val_labels = kan_evaluate(kan_model, kan_val_loader, kan_criterion)
    val_auc = roc_auc_score(val_labels, val_probs) if len(np.unique(val_labels)) > 1 else 0

    kan_scheduler.step(val_loss)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_kan_state = {k: v.clone() for k, v in kan_model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/50 | Train Loss: {train_loss:.4f} | Val AUC: {val_auc:.4f}')

    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

# Load best model
kan_model.load_state_dict(best_kan_state)
print(f"Best validation AUC: {best_val_auc:.4f}")

# Validation predictions (for recalibration)
_, _, kan_val_proba, _ = kan_evaluate(kan_model, kan_val_loader, kan_criterion)
val_predictions['kan'] = kan_val_proba

# External test — SINGLE evaluation after training complete
_, _, kan_ext_proba, kan_ext_labels = kan_evaluate(kan_model, kan_ext_loader, kan_criterion)
ext_predictions['kan'] = kan_ext_proba
kan_ext_pred = (kan_ext_proba > 0.5).astype(int)

print(f"\n=== KAN External Test ===")
print(f"AUC: {roc_auc_score(y_ext, kan_ext_proba):.4f}")
print(f"F1: {f1_score(y_ext, kan_ext_pred):.4f}")
print(f"MCC: {matthews_corrcoef(y_ext, kan_ext_pred):.4f}")

pd.DataFrame({'True_Labels': y_ext.values, 'Predicted_Probabilities': kan_ext_proba}
).to_csv(os.path.join(OUT_DIR, 'kan_results.csv'), index=False)
torch.save({'model_state_dict': kan_model.state_dict(),
            'feature_names': all_features}, os.path.join(OUT_DIR, 'kan_model.pth'))

In [ ]:
print("Computing KAN SHAP values...")
def kan_predict_fn(X):
    kan_model.eval()
    with torch.no_grad():
        t = torch.FloatTensor(X).to(device)
        return torch.softmax(kan_model(t), dim=1)[:, 1].cpu().numpy()

kan_bg = shap.sample(X_tr.values, 50, random_state=42)
kan_explainer = shap.KernelExplainer(kan_predict_fn, kan_bg, link="logit")
kan_sample = X_ext.sample(n=min(50, len(X_ext)), random_state=42)
kan_shap = kan_explainer.shap_values(kan_sample.values, nsamples=100)

plt.figure(figsize=(12, 8))
shap.summary_plot(kan_shap, kan_sample.values, feature_names=all_features, show=False, max_display=20)
plt.title('KAN — SHAP Summary')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'shap_kan.png'), dpi=300, bbox_inches='tight')
plt.close()
print("KAN SHAP saved.")

## PVC Burden Sensitivity Analysis (Major 5)

In [ ]:
# Features without PVC burden
features_no_pvc = [f for f in numeric_features if f != 'PVCyüzdesi'] + categorical_features
print(f"Features without PVC burden: {len(features_no_pvc)} features")

# LR without PVC burden
print("\nTraining LR without PVC burden...")
lr_nopvc = LogisticRegression()
lr_search_nopvc = RandomizedSearchCV(lr_nopvc, lr_params, n_iter=100, cv=5,
                                      scoring='roc_auc', random_state=42, n_jobs=-1)
lr_search_nopvc.fit(X_tr[features_no_pvc], y_tr)
lr_nopvc_ext = lr_search_nopvc.predict_proba(X_ext[features_no_pvc])[:, 1]
lr_nopvc_auc = roc_auc_score(y_ext, lr_nopvc_ext)

# XGBoost without PVC burden
print("Training XGBoost without PVC burden...")
xgb_nopvc = XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False)
xgb_search_nopvc = RandomizedSearchCV(xgb_nopvc, xgb_params, n_iter=100, cv=5,
                                       scoring='roc_auc', random_state=42, n_jobs=-1)
xgb_search_nopvc.fit(X_tr[features_no_pvc], y_tr)
xgb_nopvc_ext = xgb_search_nopvc.predict_proba(X_ext[features_no_pvc])[:, 1]
xgb_nopvc_auc = roc_auc_score(y_ext, xgb_nopvc_ext)

# Best DL model without PVC burden (KAN)
print("Training KAN without PVC burden...")
kan_nopvc_model = KolmogorovArnoldNetwork(
    input_dim=len(features_no_pvc), hidden_dims=[94, 55], inner_dim=37, dropout=0.467
).to(device)
kan_nopvc_opt = optim.Adam(kan_nopvc_model.parameters(), lr=0.001)
kan_nopvc_crit = nn.CrossEntropyLoss()
kan_nopvc_sched = optim.lr_scheduler.ReduceLROnPlateau(kan_nopvc_opt, mode='min', factor=0.5, patience=5)

nopvc_train_ds = KANDataset(X_tr[features_no_pvc], y_tr)
nopvc_val_ds = KANDataset(X_val[features_no_pvc], y_val)
nopvc_ext_ds = KANDataset(X_ext[features_no_pvc], y_ext)
nopvc_train_loader = DataLoader(nopvc_train_ds, batch_size=32, shuffle=True)
nopvc_val_loader = DataLoader(nopvc_val_ds, batch_size=32)
nopvc_ext_loader = DataLoader(nopvc_ext_ds, batch_size=32)

best_nopvc_auc = 0
best_nopvc_state = None
for epoch in range(50):
    kan_train_epoch(kan_nopvc_model, nopvc_train_loader, kan_nopvc_crit, kan_nopvc_opt)
    _, _, vp, vl = kan_evaluate(kan_nopvc_model, nopvc_val_loader, kan_nopvc_crit)
    vauc = roc_auc_score(vl, vp) if len(np.unique(vl)) > 1 else 0
    kan_nopvc_sched.step(1 - vauc)
    if vauc > best_nopvc_auc:
        best_nopvc_auc = vauc
        best_nopvc_state = {k: v.clone() for k, v in kan_nopvc_model.state_dict().items()}

kan_nopvc_model.load_state_dict(best_nopvc_state)
_, _, kan_nopvc_ext, _ = kan_evaluate(kan_nopvc_model, nopvc_ext_loader, kan_nopvc_crit)
kan_nopvc_auc = roc_auc_score(y_ext, kan_nopvc_ext)

# Report
print(f"\n=== PVC Burden Sensitivity Analysis ===")
print(f"LR:      AUC with PVC = {roc_auc_score(y_ext, ext_predictions['lr']):.4f}, without = {lr_nopvc_auc:.4f}, delta = {roc_auc_score(y_ext, ext_predictions['lr']) - lr_nopvc_auc:+.4f}")
print(f"XGBoost: AUC with PVC = {roc_auc_score(y_ext, ext_predictions['xgboost']):.4f}, without = {xgb_nopvc_auc:.4f}, delta = {roc_auc_score(y_ext, ext_predictions['xgboost']) - xgb_nopvc_auc:+.4f}")
print(f"KAN:     AUC with PVC = {roc_auc_score(y_ext, ext_predictions['kan']):.4f}, without = {kan_nopvc_auc:.4f}, delta = {roc_auc_score(y_ext, ext_predictions['kan']) - kan_nopvc_auc:+.4f}")

pd.DataFrame({
    'True_Labels': y_ext.values,
    'LR_no_PVC': lr_nopvc_ext,
    'XGBoost_no_PVC': xgb_nopvc_ext,
    'KAN_no_PVC': kan_nopvc_ext
}).to_csv(os.path.join(SENS_DIR, 'pvc_sensitivity_results.csv'), index=False)
print("Sensitivity results saved.")

## Recalibration (Major 6)

Logistic recalibration on Istanbul 20% validation set. Same validation set used for early stopping — acceptable because recalibration fits only 2 parameters (intercept + slope).

In [ ]:
from sklearn.linear_model import LogisticRegression as LR_recalib

model_names_recalib = ['lr', 'mlp', 'xgboost', 'tabtransformer', 'tabnet', 'kan']

print("=== Logistic Recalibration (on Istanbul validation set) ===\n")
for mname in model_names_recalib:
    pred_val = val_predictions[mname]
    pred_ext = ext_predictions[mname]

    # Clip extreme values
    pred_val_c = np.clip(pred_val, 1e-8, 1-1e-8)
    pred_ext_c = np.clip(pred_ext, 1e-8, 1-1e-8)

    # Logistic recalibration — fit on Istanbul validation
    recalib = LR_recalib()
    recalib.fit(sp_logit(pred_val_c).reshape(-1, 1), y_val)

    # Apply to external
    pred_ext_recalib = recalib.predict_proba(sp_logit(pred_ext_c).reshape(-1, 1))[:, 1]

    auc_before = roc_auc_score(y_ext, pred_ext)
    auc_after = roc_auc_score(y_ext, pred_ext_recalib)
    print(f"{mname:18s} | AUC before: {auc_before:.4f} | AUC after recalib: {auc_after:.4f}")

    # Save recalibrated predictions
    pd.DataFrame({
        'True_Labels': y_ext.values,
        'Predicted_Probabilities': pred_ext_recalib
    }).to_csv(os.path.join(OUT_DIR, f'{mname}_recalibrated_results.csv'), index=False)

print("\nAll recalibrated results saved.")

## Summary — All Model Results

In [ ]:
print("=" * 70)
print("FINAL EXTERNAL TEST RESULTS (Anatolia, N={})".format(len(y_ext)))
print("=" * 70)
print(f"{'Model':<20} {'AUC':>8} {'F1':>8} {'MCC':>8} {'BalAcc':>8}")
print("-" * 70)

for mname, proba in ext_predictions.items():
    pred = (proba > 0.5).astype(int)
    auc = roc_auc_score(y_ext, proba)
    f1 = f1_score(y_ext, pred)
    mcc = matthews_corrcoef(y_ext, pred)
    bacc = balanced_accuracy_score(y_ext, pred)
    print(f"{mname:<20} {auc:>8.4f} {f1:>8.4f} {mcc:>8.4f} {bacc:>8.4f}")

print("\n" + "=" * 70)
print("Files saved in: " + OUT_DIR)
print("Figures saved in: " + FIG_DIR)
import glob
print("\nCSV files:")
for f in sorted(glob.glob(os.path.join(OUT_DIR, '*.csv'))):
    print(f"  {os.path.basename(f)}")